Accessing IPython cluster clients and printing their ids to check.

In [35]:
from ipyparallel import Client

clients = Client()
clients.block = True

print(clients.ids)

[0, 1, 2]


Importing mpi4py and numpy.

In [36]:
%%px
from mpi4py import MPI
import numpy as np

Implementing Convolution, you don't need to modify this code.

In [37]:
%%px

def convolve_func(main, kernel, KERNEL_DIM, DIMx, DIMy, upper_pad, lower_pad):

    num_pads = int((KERNEL_DIM - 1) / 2)

    conv = np.zeros(main.shape, dtype=int)

    main = np.concatenate((upper_pad, main, lower_pad))

    for j in range(DIMx):
        for i in range(DIMy):

            for k in range(KERNEL_DIM):
                for l in range(KERNEL_DIM):

                    row_idx = j + k
                    col_idx = i + l - num_pads

                    if (row_idx < DIMx + 2*num_pads and
                        0 <= col_idx < DIMy):

                        conv[j*DIMy+i] += (
                            main[row_idx*DIMy+col_idx] * kernel[k][l]
                        )

    return conv

5 points: 
Load MPI communicator, get the total number of processes and rank of the process                              

In [38]:
%%px

comm = MPI.COMM_WORLD

size = comm.Get_size()
rank = comm.Get_rank()

print("Process rank:", rank, "Total processes:", size)

[stdout:0] Process rank: 0 Total processes: 3


[stdout:2] Process rank: 2 Total processes: 3


[stdout:1] Process rank: 1 Total processes: 3


5 points: 
Load or initialize data array and kernel array only in process 0(rank 0)                                      

In [39]:
%%px

DIMx = 0
DIMy = 0
KERNEL_DIM = 0

if rank == 0:

    img = np.array([
        [3, 9, 5, 9],
        [1, 7, 4, 3],
        [2, 1, 6, 5],
        [3, 9, 5, 9],
        [1, 7, 4, 3],
        [2, 1, 6, 5]
    ])

    kernel = np.array([
        [0, 1, 0],
        [0, 0, 0],
        [0, -1, 0]
    ])

    DIMx = img.shape[0]
    DIMy = img.shape[1]
    KERNEL_DIM = int(kernel.shape[0])

10 points: 
Broadcast data and kernel array sizes from process 0 to  all other processes                                 

In [40]:
%%px

dims = np.array([DIMx, DIMy, KERNEL_DIM], dtype=np.int32)

comm.Bcast(dims, root=0)

DIMx = int(dims[0])
DIMy = int(dims[1])
KERNEL_DIM = int(dims[2])

print("Rank:", rank, "Dimensions:", DIMx, DIMy, KERNEL_DIM)

[stdout:0] Rank: 0 Dimensions: 6 4 3


[stdout:2] Rank: 2 Dimensions: 6 4 3


[stdout:1] Rank: 1 Dimensions: 6 4 3


Initialize empty kernel array for all  processes except rank = 0, why we are not initialzing kernel array for rank 0?

Ans: Process 0 already contains the original kernel array. Therefore, we only need to initialize empty kernel arrays on the other processes to receive the kernel data through MPI broadcast.

In [41]:
%%px

if rank != 0:
    kernel = np.empty((KERNEL_DIM, KERNEL_DIM), dtype=np.int64)

10 points: 
Broadcast Kernel array from rank 0 to all other processes.                                                   

In [42]:
%%px

comm.Bcast(kernel, root=0)

print("Rank:", rank, "Kernel:\n", kernel)

[stdout:1] Rank: 1 Kernel:
 [[ 0  1  0]
 [ 0  0  0]
 [ 0 -1  0]]


[stdout:0] Rank: 0 Kernel:
 [[ 0  1  0]
 [ 0  0  0]
 [ 0 -1  0]]


[stdout:2] Rank: 2 Kernel:
 [[ 0  1  0]
 [ 0  0  0]
 [ 0 -1  0]]


25 points: 
Split the rows in data array equally and scatter them from process 0 to all other process. To split them 
equally, number of rows in the data array must be a integral multiple of number of processes. MPI has ways 
to send unequal chunks of data between processses. But for here you can do with equal number.

In [43]:
%%px

# Number of rows assigned to each process
rows_per_process = DIMx // size

# Allocate receive buffer on ALL processes, including rank 0
recvbuf = np.empty((rows_per_process, DIMy), dtype=np.int64)

# Only process 0 has the original input matrix
if rank == 0:
    sendbuf = img
else:
    sendbuf = None

# Scatter equal portions of the matrix
comm.Scatter(sendbuf, recvbuf, root=0)

print("Rank:", rank)
print("Received data:\n", recvbuf)

[stdout:2] Rank: 2
Received data:
 [[1 7 4 3]
 [2 1 6 5]]


[stdout:0] Rank: 0
Received data:
 [[3 9 5 9]
 [1 7 4 3]]


[stdout:1] Rank: 1
Received data:
 [[2 1 6 5]
 [3 9 5 9]]


25 points: 
For convolution of kernel array and data array, you have to pass the kernel padding rows from one
process to another. please see objective for more details. Send and Recieve rows from one process 
to other. Careful with the data size and tags you are sending and receiving should match otherwise
commincator will wait for them indefintely.                                                                  

In [44]:
%%px

upper_pad = np.zeros(DIMy, dtype=np.int64)
lower_pad = np.zeros(DIMy, dtype=np.int64)

if rank > 0:
    comm.Send(recvbuf[0, :].copy(), dest=rank-1, tag=11)
    comm.Recv(upper_pad, source=rank-1, tag=12)

if rank < size-1:
    comm.Send(recvbuf[-1, :].copy(), dest=rank+1, tag=12)
    comm.Recv(lower_pad, source=rank+1, tag=11)

print("Rank:", rank)
print("Upper padding:", upper_pad)
print("Lower padding:", lower_pad)

[stdout:0] Rank: 0
Upper padding: [0 0 0 0]
Lower padding: [2 1 6 5]


[stdout:1] Rank: 1
Upper padding: [1 7 4 3]
Lower padding: [1 7 4 3]


[stdout:2] Rank: 2
Upper padding: [3 9 5 9]
Lower padding: [0 0 0 0]


Why we are loading data into process 0 and broadcasting input data to all other processes? are there any other methods to load data into all processes (not for evaluation)

Ans: Process 0 loads the input data because it acts as the root process and coordinates data distribution. This avoids redundant loading of the complete input matrix by every process. The input matrix is distributed using MPI Scatter, while the kernel and dimensions are shared using MPI Broadcast. Alternatively, each process could independently read its portion from a shared file, or MPI-IO could be used for parallel data loading.

5 points: 
Perform Convolution operation by calling convolve_func() provided for each of the process with 
corresponding rows as arguments.                                                                             

In [45]:
%%px

local_conv = convolve_func(
    recvbuf.flatten(),
    kernel,
    KERNEL_DIM,
    rows_per_process,
    DIMy,
    upper_pad,
    lower_pad
)

print("Rank:", rank)
print("Local convolution:\n", local_conv)

[stdout:1] Rank: 1
Local convolution:
 [-2 -2 -1 -6  1 -6  2  2]


[stdout:2] Rank: 2
Local convolution:
 [ 1  8 -1  4  1  7  4  3]


[stdout:0] Rank: 0
Local convolution:
 [-1 -7 -4 -3  1  8 -1  4]


10 points: 
Gather the computed convolutional matrix rows to process 0.                                                 

In [46]:
%%px
#To receive data from all processes, process 0 should have a buffer
if rank == 0:
    gathered = np.empty(DIMx * DIMy, dtype=np.int64)
else:
    gathered = None

comm.Gather(local_conv, gathered, root=0)

if rank == 0:
    print("Gathered output:", gathered)

[stdout:0] Gathered output: [-1 -7 -4 -3  1  8 -1  4 -2 -2 -1 -6  1 -6  2  2  1  8 -1  4  1  7  4  3]


Reshape the flattened array to match input dimensions

In [47]:
%%px

if rank == 0:
    recvbuf = gathered.reshape(DIMx, DIMy)
    print("Final parallel convolution:\n", recvbuf)

[stdout:0] Final parallel convolution:
 [[-1 -7 -4 -3]
 [ 1  8 -1  4]
 [-2 -2 -1 -6]
 [ 1 -6  2  2]
 [ 1  8 -1  4]
 [ 1  7  4  3]]


5 points: 
Test to check sequential convolution and MPI based parallel convolution outputs                               

In [48]:
%%px

if rank == 0:

    main_grid = img.flatten()

    zero_pad = np.zeros(DIMy, dtype=np.int64)

    # Sequential convolution using zero padding
    conv1 = convolve_func(
        main_grid,
        kernel,
        KERNEL_DIM,
        DIMx,
        DIMy,
        zero_pad,
        zero_pad
    )

    # Reshape into the original 6 x 4 dimensions
    conv1 = np.reshape(conv1, (DIMx, DIMy))

    print("Sequential output:\n", conv1)
    print("Parallel output:\n", recvbuf)

    print("Outputs match:", np.array_equal(conv1, recvbuf))

[stdout:0] Sequential output:
 [[-1 -7 -4 -3]
 [ 1  8 -1  4]
 [-2 -2 -1 -6]
 [ 1 -6  2  2]
 [ 1  8 -1  4]
 [ 1  7  4  3]]
Parallel output:
 [[-1 -7 -4 -3]
 [ 1  8 -1  4]
 [-2 -2 -1 -6]
 [ 1 -6  2  2]
 [ 1  8 -1  4]
 [ 1  7  4  3]]
Outputs match: True
